# Attention Mechanisms Deep Dive

Attention is the most expensive operation in an LLM — both in compute (prefill) and
memory (decode KV cache). Understanding the variants explains most of the optimisation
landscape.

This notebook covers:
1. Self-attention from scratch — what Q, K, V actually are
2. Multi-Head Attention (MHA) — the original design
3. Multi-Query Attention (MQA) — shared KV heads for faster decode
4. Grouped-Query Attention (GQA) — the practical middle ground
5. FlashAttention — tiling to avoid the memory wall
6. KV cache implications of each variant
7. Hybrid architectures — linear attention + full attention (Qwen3.5)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

## 1. Self-Attention from Scratch

Every token in the sequence needs to "look at" every other token to decide what's
relevant. Attention is the mechanism for this.

### The three projections

Given an input sequence of hidden states `X` with shape `[seq_len, d_model]`:

```
Q = X @ W_q    (Query:  "what am I looking for?")
K = X @ W_k    (Key:    "what do I contain?")
V = X @ W_v    (Value:  "what information do I provide?")
```

### The attention computation

```
Attention(Q, K, V) = softmax(Q @ K^T / √d_k) @ V

Step by step:
1. Q @ K^T  →  [seq, seq] score matrix (how much each token attends to each other)
2. / √d_k   →  scale to prevent softmax saturation
3. softmax   →  normalise scores to probabilities (each row sums to 1)
4. @ V       →  weighted sum of value vectors
```

### The causal mask

For autoregressive (left-to-right) generation, token `i` can only attend to
tokens `0..i` (not future tokens). We mask the upper triangle to -inf before softmax.

In [ ]:
# Implement attention from scratch
torch.manual_seed(42)

seq_len = 8
d_model = 64
d_k = 64  # key/query dimension

# Simulated input: 8 tokens, each a 64-dim vector
X = torch.randn(seq_len, d_model)

# Projection matrices
W_q = torch.randn(d_model, d_k) * 0.1
W_k = torch.randn(d_model, d_k) * 0.1
W_v = torch.randn(d_model, d_k) * 0.1

# Project
Q = X @ W_q  # [8, 64]
K = X @ W_k  # [8, 64]
V = X @ W_v  # [8, 64]

# Attention scores
scores = Q @ K.T / (d_k ** 0.5)  # [8, 8]

# Causal mask: token i can only see tokens 0..i
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores = scores.masked_fill(mask, float('-inf'))

# Softmax → attention weights
attn_weights = F.softmax(scores, dim=-1)  # [8, 8]

# Weighted sum of values
output = attn_weights @ V  # [8, 64]

print(f"Input shape:   {X.shape}  (seq_len, d_model)")
print(f"Q, K, V shape: {Q.shape}  (seq_len, d_k)")
print(f"Scores shape:  {scores.shape}  (seq_len, seq_len) ← this is the O(n²) part")
print(f"Output shape:  {output.shape}  (seq_len, d_k)")
print(f"\nAttention weights (each row sums to 1):")
print(attn_weights.numpy().round(3))

In [ ]:
# Visualise the attention pattern
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im1 = ax1.imshow(attn_weights.numpy(), cmap='Blues')
ax1.set_xlabel('Key position (attending to)')
ax1.set_ylabel('Query position (attending from)')
ax1.set_title('Causal attention weights\n(lower triangle only — can\'t see future)')
plt.colorbar(im1, ax=ax1)

# Show the cost: O(n²) in sequence length
seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
d = 128
flops = 2 * seq_lengths ** 2 * d  # Q@K^T + attn@V
memory_score_matrix = seq_lengths ** 2 * 2  # FP16 bytes

ax2.semilogy(seq_lengths, flops / 1e9, 'b-o', label='Compute (GFLOPs)', linewidth=2)
ax2.semilogy(seq_lengths, memory_score_matrix / 1e6, 'r-s', label='Score matrix (MB)', linewidth=2)
ax2.set_xlabel('Sequence length')
ax2.set_ylabel('Cost')
ax2.set_title('Attention cost scales quadratically with sequence length')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"At seq=32768: score matrix = {32768**2 * 2 / 1e9:.1f} GB in FP16")
print(f"This is why FlashAttention exists — to avoid materialising this matrix.")

## 2. Multi-Head Attention (MHA) — The Original Design

Instead of one big attention with `d_model=2048`, split into multiple **heads**,
each attending in a smaller subspace:

```
d_model = 2048, num_heads = 32 → head_dim = 64

For each head h (independently, in parallel):
    Q_h = X @ W_q_h   [seq, 64]
    K_h = X @ W_k_h   [seq, 64]
    V_h = X @ W_v_h   [seq, 64]
    out_h = Attention(Q_h, K_h, V_h)  [seq, 64]

Concatenate all heads: [seq, 64×32] = [seq, 2048]
Final projection: output = concat @ W_o  [seq, 2048]
```

### Why multiple heads?

Each head can learn a different type of relationship:
- Head 1: syntactic (subject-verb agreement)
- Head 2: positional (nearby tokens)
- Head 3: semantic (related concepts)
- Head 4: coreference (pronouns → antecedents)

### The KV cache cost of MHA

Every head has its own K and V. For decode, we must store:
```
KV cache per token = 2 × num_heads × head_dim × 2 bytes
                   = 2 × 32 × 128 × 2 = 16,384 bytes = 16 KB per token per layer

For 32 layers at 4096 context: 32 × 4096 × 16 KB = 2 GB
```

This is where MQA and GQA come in — they reduce the KV cache size.

In [ ]:
# Multi-head attention implementation
def multi_head_attention(X, num_heads, d_model, W_q, W_k, W_v, W_o, causal=True):
    seq_len = X.shape[0]
    head_dim = d_model // num_heads
    
    # Project and reshape to [num_heads, seq, head_dim]
    Q = (X @ W_q).reshape(seq_len, num_heads, head_dim).transpose(0, 1)
    K = (X @ W_k).reshape(seq_len, num_heads, head_dim).transpose(0, 1)
    V = (X @ W_v).reshape(seq_len, num_heads, head_dim).transpose(0, 1)
    
    # Attention per head
    scores = Q @ K.transpose(-2, -1) / (head_dim ** 0.5)
    if causal:
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    out = attn @ V  # [num_heads, seq, head_dim]
    
    # Concatenate heads and project
    out = out.transpose(0, 1).reshape(seq_len, d_model)
    return out @ W_o, K, V

# Setup
d_model = 256
num_heads = 8
seq_len = 16

X = torch.randn(seq_len, d_model)
W_q = torch.randn(d_model, d_model) * 0.02
W_k = torch.randn(d_model, d_model) * 0.02
W_v = torch.randn(d_model, d_model) * 0.02
W_o = torch.randn(d_model, d_model) * 0.02

output, K_cache, V_cache = multi_head_attention(X, num_heads, d_model, W_q, W_k, W_v, W_o)
print(f"MHA output shape: {output.shape}")
print(f"K cache shape: {K_cache.shape} (num_heads, seq_len, head_dim)")
print(f"V cache shape: {V_cache.shape}")
print(f"\nKV cache size per layer: {K_cache.numel() * 2 + V_cache.numel() * 2:.0f} bytes")
print(f"  = {num_heads} heads × {seq_len} tokens × {d_model // num_heads} dim × 2 (K+V) × 2 bytes")

## 3. Multi-Query Attention (MQA) — Shared KV for Speed

**Key insight**: During decode, loading the KV cache from HBM is the bottleneck.
What if we share K and V across all query heads?

```
MHA:  32 query heads, 32 key heads, 32 value heads
MQA:  32 query heads,  1 key head,   1 value head   ← shared!
```

Each query head still has its own Q projection (different "questions"), but they
all look at the same K and V (same "answers").

### Memory savings

```
MHA KV cache: 2 × 32 heads × seq_len × head_dim × 2 bytes
MQA KV cache: 2 ×  1 head  × seq_len × head_dim × 2 bytes

Reduction: 32x smaller KV cache!
```

### The tradeoff
- Faster decode (less KV cache to load from HBM)
- More concurrent requests fit in memory
- But: slight quality degradation (all heads share the same key/value representation)

Used by: PaLM, Falcon, StarCoder

## 4. Grouped-Query Attention (GQA) — The Practical Middle Ground

GQA groups query heads into clusters, each sharing one set of K/V heads:

```
MHA:  32 Q heads → 32 KV heads (1:1 ratio)
GQA:  32 Q heads →  8 KV heads (4:1 ratio)    ← LLaMA-3, Qwen, Mistral
MQA:  32 Q heads →  1 KV head  (32:1 ratio)

    MHA (32 KV heads)          GQA (8 KV heads)          MQA (1 KV head)
    ══════════════════          ════════════════          ════════════════
    Q₁  → K₁, V₁              Q₁ ─┐                    Q₁  ─┐
    Q₂  → K₂, V₂              Q₂ ─┤→ K₁, V₁           Q₂  ─┤
    Q₃  → K₃, V₃              Q₃ ─┤                    Q₃  ─┤
    Q₄  → K₄, V₄              Q₄ ─┘                    Q₄  ─┤
    Q₅  → K₅, V₅              Q₅ ─┐                    ...  ─┤→ K₁, V₁
    ...                        Q₆ ─┤→ K₂, V₂           ...  ─┤
    Q₃₂ → K₃₂, V₃₂           Q₇ ─┤                    Q₃₁ ─┤
                               Q₈ ─┘                    Q₃₂ ─┘
                               ...
```

### Why GQA wins

| Variant | KV heads | KV cache size (relative) | Quality | Decode speed |
|---------|----------|-------------------------|---------|-------------|
| MHA | 32 | 1.0x (baseline) | Best | Slowest |
| GQA-8 | 8 | 0.25x | Near-MHA | 4x faster KV load |
| GQA-4 | 4 | 0.125x | Good | 8x faster KV load |
| MQA | 1 | 0.03x | Noticeably worse | 32x faster KV load |

GQA-8 (used by LLaMA-3, Mistral, Qwen) gives **4x KV cache reduction** with
negligible quality loss — the sweet spot for production serving.

In [ ]:
# Compare KV cache sizes across attention variants

def kv_cache_size(num_layers, num_kv_heads, head_dim, seq_len, bytes_per_elem=2):
    """KV cache in bytes."""
    return 2 * num_layers * num_kv_heads * head_dim * seq_len * bytes_per_elem

# Model: 32 layers, 32 query heads, head_dim=128, 4096 context
configs = {
    "MHA (32 KV heads)": {"kv_heads": 32, "desc": "GPT-3, original LLaMA"},
    "GQA-8 (8 KV heads)": {"kv_heads": 8, "desc": "LLaMA-3, Mistral, Qwen"},
    "GQA-4 (4 KV heads)": {"kv_heads": 4, "desc": "Some smaller models"},
    "GQA-2 (2 KV heads)": {"kv_heads": 2, "desc": "Qwen3.5-2B full-attn layers"},
    "MQA (1 KV head)": {"kv_heads": 1, "desc": "PaLM, Falcon"},
}

num_layers = 32
head_dim = 128
seq_len = 4096

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(configs.keys())
sizes_mb = [kv_cache_size(num_layers, c["kv_heads"], head_dim, seq_len) / 1e6 for c in configs.values()]
colors = ['red', 'green', 'blue', 'purple', 'orange']

bars = ax1.bar(range(len(names)), sizes_mb, color=colors, alpha=0.7)
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels([n.split(' (')[0] for n in names], rotation=15)
ax1.set_ylabel('KV cache size (MB)')
ax1.set_title(f'KV cache per request at {seq_len} tokens\n(32 layers, head_dim=128)')
ax1.grid(True, alpha=0.3, axis='y')
for bar, size in zip(bars, sizes_mb):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{size:.0f} MB', ha='center', fontsize=10)

# How many concurrent requests fit in 80GB (A100) with 14GB model
available_memory_gb = 80 - 14  # A100 minus model weights
max_requests = [available_memory_gb * 1e3 / s for s in sizes_mb]

bars2 = ax2.bar(range(len(names)), max_requests, color=colors, alpha=0.7)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels([n.split(' (')[0] for n in names], rotation=15)
ax2.set_ylabel('Max concurrent requests')
ax2.set_title(f'Concurrent requests in 66 GB free memory\n(A100 80GB minus 14GB model)')
ax2.grid(True, alpha=0.3, axis='y')
for bar, n in zip(bars2, max_requests):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{n:.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"GQA-8 vs MHA: {sizes_mb[0]/sizes_mb[1]:.0f}x less KV cache = {max_requests[1]/max_requests[0]:.0f}x more concurrent requests")
print(f"\nThis is why every modern LLM uses GQA — it's free throughput.")

## 5. FlashAttention — Tiling to Avoid the Memory Wall

### The problem with naive attention

Standard attention materialises the full `[seq, seq]` score matrix in HBM:
1. Compute `S = Q @ K^T` → write [seq, seq] to HBM
2. Read `S` from HBM → compute softmax → write `P` to HBM
3. Read `P` from HBM → compute `P @ V` → write output to HBM

That's **3 round-trips** to HBM for a matrix that's `seq² × 2` bytes.
At seq=8192: `8192² × 2 = 128 MB` per head per layer.

### FlashAttention's solution: tile everything into SRAM

```
Standard:                              FlashAttention:
                                       
Q ──┐                                  Q tiles ──┐
    ├──→ S (in HBM) ──→ softmax        K tiles ──┼──→ partial scores (in SRAM!)
K ──┘         │              │         V tiles ──┘         │
              ▼              ▼                              ▼
         P (in HBM) ──→ P @ V               online softmax + accumulate
              │                                            │
              ▼                                            ▼
         output (HBM)                               output (HBM)
              
  3 HBM round-trips                      1 HBM round-trip
  O(seq²) memory                         O(seq) memory
```

### The online softmax trick

Normal softmax requires seeing ALL scores before normalising. FlashAttention uses
an **online** algorithm that processes tiles incrementally:

```python
# Pseudocode for one query tile
m = -inf     # running max (for numerical stability)
l = 0        # running sum of exp(scores)
o = 0        # running output accumulator

for each K_tile, V_tile:
    s = Q_tile @ K_tile^T          # compute in SRAM
    m_new = max(m, s.max())        # update max
    p = exp(s - m_new)             # local softmax numerator
    l = l * exp(m - m_new) + p.sum()  # correct running sum
    o = o * exp(m - m_new) + p @ V_tile  # correct running output
    m = m_new

output = o / l                     # final normalisation
```

Each tile is computed entirely in SRAM. The score matrix **never exists in HBM**.

### Impact

| | Standard Attention | FlashAttention |
|---|---|---|
| HBM memory | O(seq²) | O(seq) |
| HBM reads/writes | 3 × seq² | ~seq (Q,K,V in, output out) |
| Speed (long seq) | Slow (HBM-bound) | 2-4x faster |
| Max sequence length | Limited by GPU memory | Much longer |
| Exact? | Yes | Yes (not an approximation!) |

In [ ]:
# Compare memory usage: standard vs flash (simulated)

seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
head_dim = 128
num_heads = 32

standard_memory_mb = []
flash_memory_mb = []

for seq in seq_lengths:
    # Standard: must store full [seq, seq] score matrix per head
    score_matrix = seq * seq * 2  # FP16
    qkv = 3 * seq * head_dim * 2  # Q, K, V
    standard_memory_mb.append((score_matrix * num_heads + qkv * num_heads) / 1e6)
    
    # Flash: only Q, K, V, output + small SRAM tiles
    qkvo = 4 * seq * head_dim * 2  # Q, K, V, O
    flash_memory_mb.append((qkvo * num_heads) / 1e6)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(seq_lengths, standard_memory_mb, 'r-o', linewidth=2, label='Standard attention')
ax.semilogy(seq_lengths, flash_memory_mb, 'g-o', linewidth=2, label='FlashAttention')
ax.axhline(y=80000, color='gray', linestyle='--', alpha=0.5, label='A100 80GB')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Memory usage (MB, log scale)')
ax.set_title('Attention memory: Standard (O(n²)) vs FlashAttention (O(n))')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"At seq=32768:")
print(f"  Standard: {standard_memory_mb[-2]:.0f} MB (score matrices alone)")
print(f"  Flash:    {flash_memory_mb[-2]:.0f} MB")
print(f"  Savings:  {standard_memory_mb[-2]/flash_memory_mb[-2]:.0f}x less memory")

## 6. KV Cache Implications

The attention variant directly determines KV cache cost during serving:

```
KV cache per token per layer = 2 × num_kv_heads × head_dim × bytes

For a 70B model (80 layers):

MHA (32 KV heads, d=128):  2 × 32 × 128 × 2 bytes = 16 KB/token/layer
                           × 80 layers = 1.28 MB per token
                           × 4096 context = 5.2 GB per request

GQA-8 (8 KV heads, d=128): 2 × 8 × 128 × 2 bytes = 4 KB/token/layer
                            × 80 layers = 320 KB per token
                            × 4096 context = 1.3 GB per request
```

### The serving economics

On an A100-80GB with a 70B model (35GB in INT4):
- Available for KV cache: ~45 GB
- MHA: 45 GB / 5.2 GB = **8 concurrent requests**
- GQA-8: 45 GB / 1.3 GB = **34 concurrent requests**

**4x more concurrent requests = 4x higher throughput** (since decode is memory-bound,
batching more requests together barely increases latency).

### KV cache quantisation compounds the benefit

GQA-8 + INT4 KV cache:
```
2 × 8 × 128 × 0.5 bytes = 1 KB/token/layer × 80 = 80 KB per token
× 4096 context = 320 MB per request
→ 45 GB / 0.32 GB = 140 concurrent requests!
```

## 7. Hybrid Architectures — Linear + Full Attention

Qwen3.5 (the model in our notebook 01) uses a **hybrid** approach:
- 18 layers with **linear attention** (Gated Delta Net)
- 6 layers with **full quadratic attention** (GQA-2)

### Why hybrid?

| | Full attention | Linear attention |
|---|---|---|
| Complexity | O(n²) in seq length | O(n) — constant memory per token |
| Quality | Best (sees all pairs) | Good (compresses history into state) |
| KV cache | Grows with sequence | Fixed size (recurrent state) |
| Long context | Memory-limited | Unlimited (in theory) |

### Linear attention: how it avoids O(n²)

Standard attention: `softmax(Q @ K^T) @ V` requires the full `[seq, seq]` matrix.

Linear attention replaces softmax with a kernel function φ:
```
Standard:  output = softmax(Q @ K^T) @ V          → O(seq²)
Linear:    output = φ(Q) @ (φ(K)^T @ V)           → O(seq × d²)
                          └── recurrent state ──┘
```

The recurrent state `φ(K)^T @ V` is a fixed-size `[d, d]` matrix that summarises
the entire history. New tokens update it incrementally — no need to store per-token K/V.

### Qwen3.5-2B cache structure

```
Layer 0 (linear):  recurrent state [d, d] + conv state [kernel_size]
Layer 1 (linear):  recurrent state [d, d] + conv state
Layer 2 (linear):  recurrent state [d, d] + conv state
Layer 3 (full):    KV cache [seq_len, num_kv_heads, head_dim]  ← grows with context!
Layer 4 (linear):  recurrent state [d, d] + conv state
...repeating...
```

Only 6 out of 24 layers have growing KV caches. The other 18 have fixed-size state.
This dramatically reduces memory pressure at long contexts compared to a pure transformer.

### The design space

| Architecture | Examples | KV cache scaling | Quality |
|---|---|---|---|
| All full attention | GPT-4, LLaMA-3 | O(layers × seq) | Best |
| All linear (Mamba/RWKV) | Mamba-2, RWKV-6 | O(layers) — constant! | Good, worse on recall |
| Hybrid | Qwen3.5, Jamba, Zamba | O(full_layers × seq) + O(linear_layers) | Near-best |

In [ ]:
# Compare KV cache growth: pure transformer vs hybrid vs pure linear

seq_lengths = np.array([256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
num_layers = 24
head_dim = 128

# Pure transformer (all GQA-8)
pure_transformer_mb = [kv_cache_size(num_layers, 8, head_dim, s) / 1e6 for s in seq_lengths]

# Hybrid (6 full-attention GQA-2 layers + 18 linear layers with fixed state)
linear_state_size = 18 * 2 * 128 * 128 * 2  # 18 layers × [d, d] state × K+V × FP16
hybrid_mb = [
    (kv_cache_size(6, 2, 256, s) + linear_state_size) / 1e6  # Qwen3.5 config
    for s in seq_lengths
]

# Pure linear (all recurrent, no KV cache growth)
pure_linear_mb = [num_layers * 2 * 128 * 128 * 2 / 1e6] * len(seq_lengths)  # constant!

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogy(seq_lengths, pure_transformer_mb, 'r-o', linewidth=2, label='Pure transformer (24L × GQA-8)')
ax.semilogy(seq_lengths, hybrid_mb, 'g-s', linewidth=2, label='Hybrid: 6 full + 18 linear (Qwen3.5-2B)')
ax.semilogy(seq_lengths, pure_linear_mb, 'b-^', linewidth=2, label='Pure linear (Mamba-style)')
ax.axhline(y=80000, color='gray', linestyle='--', alpha=0.5, label='80 GB GPU memory')
ax.set_xlabel('Sequence length')
ax.set_ylabel('Cache memory (MB, log scale)')
ax.set_title('Cache memory scaling by architecture type')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
plt.tight_layout()
plt.show()

print(f"At 128K tokens:")
print(f"  Pure transformer: {pure_transformer_mb[-1]:.0f} MB")
print(f"  Hybrid (Qwen3.5): {hybrid_mb[-1]:.0f} MB")
print(f"  Pure linear:      {pure_linear_mb[-1]:.0f} MB (constant!)")
print(f"\nHybrid uses {pure_transformer_mb[-1]/hybrid_mb[-1]:.1f}x less memory than pure transformer at 128K.")

## Summary

| Concept | Key takeaway |
|---|---|
| Self-attention | Each token computes relevance scores against all others — O(n²) |
| Multi-Head (MHA) | Independent KV per head. Best quality, largest cache. |
| Multi-Query (MQA) | 1 shared KV. 32x smaller cache, noticeable quality loss. |
| Grouped-Query (GQA) | N shared KV groups. 4-8x smaller cache, negligible quality loss. **The standard.** |
| FlashAttention | Tiles attention into SRAM. Exact, 2-4x faster, O(n) memory. |
| Hybrid (linear+full) | Fixed-size recurrent state for most layers. Cache grows only at full-attention layers. |

**The optimisation path**: GQA reduces KV cache size → FlashAttention reduces HBM traffic → 
KV quantisation further compresses the cache → hybrid architectures bound cache growth entirely.